In [0]:
spark

In [0]:
spark.sql("SHOW CATALOGS").show()

In [0]:
spark.sql("""
    CREATE CATALOG IF NOT EXISTS live_traffic_kafka
    """)

In [0]:
spark.sql(
    """
    CREATE SCHEMA IF NOT EXISTS live_traffic_kafka.bronze
    """
)

spark.sql(
    """
    CREATE SCHEMA IF NOT EXISTS live_traffic_kafka.silver
    """
)

spark.sql(
    """
    CREATE SCHEMA IF NOT EXISTS live_traffic_kafka.gold
    """
)

In [0]:
spark.sql(
    "SHOW SCHEMAS IN live_traffic_kafka"
).show()

In [0]:
secrets = dbutils.secrets.list(
    catalog="live_traffic_kafka",
    schema="bronze"
)

secrets

In [0]:
%sql
SELECT COUNT(*)
FROM live_traffic_kafka.bronze.traffic_bronze;

In [0]:
%sql
SELECT * FROM live_traffic_kafka.bronze.traffic_bronze limit 5;

In [0]:
%sql
SELECT
    timestamp,
    event_type,
    level,
    origin.flow_name,
    details:flow_progress:status AS flow_status,
    details:flow_progress:metrics:num_output_rows AS num_output_rows,
    details:flow_progress:metrics:backlog_records AS backlog_records,
    details:flow_progress:metrics:backlog_bytes AS backlog_bytes,
    details:flow_progress:metrics:source_metrics AS source_metrics
FROM event_log(
    'af712c6e-6484-4470-b985-9d7924b3efbd'
)
WHERE event_type = 'flow_progress'
  AND origin.flow_name IS NOT NULL
  AND origin.flow_name != 'pipelines.flowTimeMetrics.missingFlowName'
ORDER BY timestamp DESC
LIMIT 20;